# Cell 1: Import Library

In [1]:
import tensorflow as tf
import numpy as np
import cv2
import os
import pickle
from sklearn.metrics.pairwise import cosine_similarity

# Jika notebook dijalankan dari dalam folder 'notebooks', pindah ke root proyek
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
    print(f"Working directory diubah ke: {os.getcwd()}")
else:
    print(f"Working directory saat ini: {os.getcwd()}")
    
print('✅ Library imported')

Working directory diubah ke: e:\AIC\BlockchainAI
✅ Library imported


# Cell 2: Load Classifier & Embedding Model

In [2]:
import tensorflow as tf
from tensorflow.keras.models import load_model

model_path = 'models/panadol_v4_clean.h5'

# Load model dengan compile=False untuk menghindari error kompatibilitas
model = load_model(model_path, compile=False)

# Ambil output dari GlobalAveragePooling2D (layer index 1)
embedding_model = tf.keras.Model(inputs=model.input, outputs=model.layers[1].output)

print('✅ Embedding model built')

TypeError: Error when deserializing class 'Dense' using config={'name': 'dense', 'trainable': True, 'dtype': 'float32', 'units': 1, 'activation': 'sigmoid', 'use_bias': True, 'kernel_initializer': {'module': 'keras.initializers', 'class_name': 'GlorotUniform', 'config': {'seed': None}, 'registered_name': None}, 'bias_initializer': {'module': 'keras.initializers', 'class_name': 'Zeros', 'config': {}, 'registered_name': None}, 'kernel_regularizer': None, 'bias_regularizer': None, 'kernel_constraint': None, 'bias_constraint': None, 'quantization_config': None}.

Exception encountered: Unrecognized keyword arguments passed to Dense: {'quantization_config': None}

# Cell 3: Fungsi crop_and_enhance dan find_medicine_package

In [ ]:
img_height, img_width = 224, 224

def crop_and_enhance(image, rect):
    # Jika rect None, gunakan seluruh gambar
    if rect is None:
        h, w = image.shape[:2]
        rect = (0, 0, w, h)

    x, y, w_rect, h_rect = rect

    pad_x = int(w_rect * 0.02)
    pad_y = int(h_rect * 0.02)
    x1 = max(0, x - pad_x)
    y1 = max(0, y - pad_y)
    x2 = min(image.shape[1], x + w_rect + pad_x)
    y2 = min(image.shape[0], y + h_rect + pad_y)

    cropped = image[y1:y2, x1:x2]
    if cropped.size == 0:
        cropped = image

    enhanced = cv2.resize(cropped, (img_height, img_width))

    kernel_sharpen = np.array([[-1, -1, -1],
                                [-1,  9, -1],
                                [-1, -1, -1]])
    enhanced = cv2.filter2D(enhanced, -1, kernel_sharpen)

    lab = cv2.cvtColor(enhanced, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    enhanced = cv2.merge([l, a, b])
    enhanced = cv2.cvtColor(enhanced, cv2.COLOR_LAB2BGR)

    return enhanced

In [ ]:
def find_medicine_package(image):
    h, w = image.shape[:2]
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    min_area = (h * w) * 0.02

    # Strategi 1: Canny
    edges = cv2.Canny(blurred, 20, 120)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    dilated = cv2.dilate(edges, kernel, iterations=4)
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    best_rect = None
    max_area = 0
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area > max_area and area > min_area:
            max_area = area
            best_rect = cv2.boundingRect(cnt)

    if best_rect is not None:
        x, y, bw, bh = best_rect
        # Jika bounding box mencakup >85% gambar, anggap tidak valid
        if (bw * bh) < (h * w) * 0.85:
            return best_rect

    # Strategi 2: Adaptive threshold
    thresh = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY_INV, 15, 3)
    kernel2 = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 9))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel2)
    contours2, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    best_rect2 = None
    max_area2 = 0
    for cnt in contours2:
        area = cv2.contourArea(cnt)
        if area > max_area2 and area > min_area:
            max_area2 = area
            best_rect2 = cv2.boundingRect(cnt)

    if best_rect2 is not None:
        x, y, bw, bh = best_rect2
        if (bw * bh) < (h * w) * 0.85:
            return best_rect2

    # Strategi 3: Otsu
    _, otsu = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    kernel3 = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))
    otsu = cv2.morphologyEx(otsu, cv2.MORPH_CLOSE, kernel3)
    contours3, _ = cv2.findContours(otsu, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    best_rect3 = None
    max_area3 = 0
    for cnt in contours3:
        area = cv2.contourArea(cnt)
        if area > max_area3 and area > min_area:
            max_area3 = area
            best_rect3 = cv2.boundingRect(cnt)

    if best_rect3 is not None:
        x, y, bw, bh = best_rect3
        if (bw * bh) < (h * w) * 0.85:
            return best_rect3

    # Tidak ada kontur valid → kemungkinan non‑obat
    return None

# Cell 4: Ekstraksi Embedding

In [ ]:
def get_embeddings(data_dir):
    emb_list = []
    for class_folder in ['asli', 'palsu']:
        folder = os.path.join(data_dir, class_folder)
        for fname in os.listdir(folder):
            if not fname.lower().endswith(('.jpg','.jpeg','.png')): continue
            img = cv2.imread(os.path.join(folder, fname))
            if img is None: continue
            rect = find_medicine_package(img)
            if rect is None: rect = (0,0,img.shape[1],img.shape[0])
            enh = crop_and_enhance(img, rect)
            rgb = cv2.cvtColor(enh, cv2.COLOR_BGR2RGB)
            arr = np.expand_dims(rgb, axis=0) / 255.0
            emb = embedding_model.predict(arr, verbose=0)[0]
            emb_list.append(emb)
    return np.array(emb_list)

print('Mengekstrak embedding dari data/dataset_clean...')
embeddings = get_embeddings('data/dataset_clean')
centroid = np.mean(embeddings, axis=0)
sims = cosine_similarity(embeddings, [centroid]).flatten()
threshold = np.percentile(sims, 5)
print(f'Centroid shape: {centroid.shape}, OOD threshold: {threshold:.4f}')

# Cell 5: Fungsi is_ood

In [ ]:
def is_ood(image_path):
    img = cv2.imread(image_path)
    if img is None: return True, 0.0
    rect = find_medicine_package(img)
    if rect is None: rect = (0,0,img.shape[1],img.shape[0])
    enh = crop_and_enhance(img, rect)
    rgb = cv2.cvtColor(enh, cv2.COLOR_BGR2RGB)
    arr = np.expand_dims(rgb, axis=0) / 255.0
    emb = embedding_model.predict(arr, verbose=0)[0]
    sim = cosine_similarity([emb], [centroid])[0][0]
    if sim < threshold:
        return True, sim
    else:
        return False, sim
print('✅ Fungsi is_ood siap')

# Cell 6: Evaluasi pada demo_test

In [ ]:
import pandas as pd
gt = pd.read_csv('output/demo_test_ground_truth.csv')
gt_clean = gt[gt['category'].isin(['obat','non_obat'])]
obat_files = gt_clean[gt_clean['category']=='obat']['filename'].tolist()
non_obat_files = gt_clean[gt_clean['category']=='non_obat']['filename'].tolist()
demo_dir = 'data/demo_test'
# Obat
rejected_obat = 0
for fname in obat_files:
    ood, sim = is_ood(os.path.join(demo_dir, fname))
    if ood: rejected_obat += 1
# Non-obat
false_pos = 0
for fname in non_obat_files:
    ood, sim = is_ood(os.path.join(demo_dir, fname))
    if not ood: false_pos += 1
recall = 1 - (rejected_obat / len(obat_files)) if obat_files else 0
print(f'Recall: {recall:.2%}, False Positive: {false_pos}')

# Cell 7: Simpan Resources

In [ ]:
with open('output/ood_resources.pkl', 'wb') as f:
    pickle.dump({'centroid': centroid, 'threshold': threshold}, f)
print('✅ OOD resources saved to output/ood_resources.pkl')